In [ ]:
# ============================================================================
# CELL 1: IMPORTS, CONFIG, MODEL DEFINITION
# ============================================================================

import os
import json
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from tqdm import tqdm
import sentencepiece as spm
from datetime import datetime
import math
import csv

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    # Paths
    CHECKPOINT_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-models\bpe16_updated_19k.pt"
    TOKENIZER_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"
    TRAIN_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Filtered\train_plus_val_filtered.jsonl"
    TEST_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Filtered\test_filtered.jsonl"
    OUTPUT_DIR = "finetuned-summarization"
    
    # Model settings
    BLOCK_SIZE = 1024
    MAX_SOURCE_LENGTH = 768
    MAX_TARGET_LENGTH = 256
    
    # Freezing strategy - UPDATED: Freeze blocks 0-6 (train blocks 7-11)
    FREEZE_EMBEDDINGS = False
    FREEZE_BLOCKS = list(range(7))  # Freeze blocks 0-6, train 7-11
    
    # Training hyperparameters
    NUM_EPOCHS = 10
    BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 4  # UPDATED: Effective batch size = 16
    LEARNING_RATE = 1e-5
    WARMUP_RATIO = 0.1
    MAX_GRAD_NORM = 1.0
    WEIGHT_DECAY = 0.01
    
    # Generation settings - UPDATED
    GENERATION_TOP_K = 50  # Increased from 40
    GENERATION_TOP_P = 0.9  # NEW: Nucleus sampling
    GENERATION_TEMPERATURE = 0.5  # Reduced from 0.7 for focused summaries
    MAX_NEW_TOKENS = 64
    
    # Evaluation - UPDATED
    EVAL_STEPS = 500  # Reduced frequency from 100
    ROUGE_SAMPLE_SIZE = 50  # Reduced from 100 for faster eval
    LOG_STEPS = 10
    
    # Early stopping - NEW
    EARLY_STOPPING_PATIENCE = 3  # Stop if no improvement for 3 epochs
    
    # Safety
    MAX_CONSECUTIVE_NANS = 3

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 16384
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, input_ids, labels=None):
        B, T = input_ids.size()
        assert T <= self.config.block_size, f"Sequence length {T} exceeds block size {self.config.block_size}"
        
        pos = torch.arange(0, T, dtype=torch.long, device=input_ids.device)
        pos_emb = self.transformer.wpe(pos)
        tok_emb = self.transformer.wte(input_ids)
        x = tok_emb + pos_emb
        
        for block in self.transformer.h:
            x = block(x)
        
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        
        return loss, logits

# ============================================================================
# LOAD MODEL & APPLY FREEZING
# ============================================================================

print("\n" + "="*80)
print("LOADING MODEL")
print("="*80)

sp = spm.SentencePieceProcessor()
sp.load(config.TOKENIZER_PATH)
print(f"✓ Tokenizer loaded (vocab: {sp.vocab_size()})")

checkpoint = torch.load(config.CHECKPOINT_PATH, map_location=device, weights_only=False)
model_config = checkpoint['config']
model = GPT(model_config)
model.load_state_dict(checkpoint['model'])
model.to(device)
print(f"✓ Model loaded (step: {checkpoint['step']}, val_loss: {checkpoint['val_loss']:.4f})")

# Apply freezing strategy
print("\n" + "="*80)
print("APPLYING FREEZING STRATEGY")
print("="*80)

if config.FREEZE_EMBEDDINGS:
    for param in model.transformer.wte.parameters():
        param.requires_grad = False
    for param in model.transformer.wpe.parameters():
        param.requires_grad = False
    print("✓ Frozen: Token & Position Embeddings")
else:
    for param in model.transformer.wpe.parameters():
        param.requires_grad = False
    print("✓ Frozen: Position Embeddings only")
    print("✓ Token Embeddings (wte) remain trainable (shares weights with lm_head)")

for block_idx in config.FREEZE_BLOCKS:
    for param in model.transformer.h[block_idx].parameters():
        param.requires_grad = False
print(f"✓ Frozen: Blocks {config.FREEZE_BLOCKS[0]}-{config.FREEZE_BLOCKS[-1]}")
print(f"✓ Trainable: Blocks {config.FREEZE_BLOCKS[-1]+1}-{model_config.n_layer-1} + Final LayerNorm + LM Head")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n✓ Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

print("\n✓ CELL 1 COMPLETE: Model loaded and configured")

In [ ]:
# ============================================================================
# CELL 2: DATASET LOADING & PREPARATION
# ============================================================================

print("\n" + "="*80)
print("LOADING DATASET")
print("="*80)

# Nepali prompt template
PROMPT_TEMPLATE = "यो लेखको संक्षेप गर्नुहोस्:\n{text}\nसारांश:\n"

class SummarizationDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, block_size):
        self.tokenizer = tokenizer
        self.block_size = block_size
        
        self.data = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    item = json.loads(line)
                    if 'text' in item and 'summary' in item:
                        self.data.append(item)
        
        print(f"  Loaded {len(self.data)} samples from {os.path.basename(jsonl_path)}")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Build full sequence
        prompt = PROMPT_TEMPLATE.format(text=item['text'])
        full_text = prompt + item['summary']
        
        # Tokenize
        tokens = self.tokenizer.encode(full_text)
        
        # Safety check: truncate if exceeds block size
        if len(tokens) > self.block_size:
            tokens = tokens[:self.block_size]
        
        input_ids = torch.tensor(tokens, dtype=torch.long)
        labels = input_ids.clone()
        
        # Mask prompt tokens in labels (only compute loss on summary)
        prompt_len = len(self.tokenizer.encode(prompt))
        labels[:prompt_len] = -100
        
        return {
            'input_ids': input_ids,
            'labels': labels,
            'text': item['text'],
            'summary': item['summary']
        }

def collate_fn(batch):
    """Collate function with dynamic padding"""
    max_len = max(len(item['input_ids']) for item in batch)
    
    input_ids = []
    labels = []
    
    for item in batch:
        seq_len = len(item['input_ids'])
        pad_len = max_len - seq_len
        
        input_ids.append(torch.cat([item['input_ids'], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([item['labels'], torch.full((pad_len,), -100, dtype=torch.long)]))
    
    return {
        'input_ids': torch.stack(input_ids),
        'labels': torch.stack(labels)
    }

# Load datasets
train_dataset = SummarizationDataset(config.TRAIN_JSONL, sp, config.BLOCK_SIZE)
test_dataset = SummarizationDataset(config.TEST_JSONL, sp, config.BLOCK_SIZE)

print(f"\n✓ Train samples: {len(train_dataset)}")
print(f"✓ Test samples:  {len(test_dataset)}")

# Create dataloaders with workers for efficiency
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)

print(f"✓ Train batches per epoch: {len(train_loader)}")
print(f"✓ Optimizer steps per epoch: ~{len(train_loader) // config.GRADIENT_ACCUMULATION_STEPS}")

print("\n✓ CELL 2 COMPLETE: Datasets loaded")

In [ ]:
# ============================================================================
# CELL 3: TRAINING LOOP
# ============================================================================

print("\n" + "="*80)
print("OPTIMIZER SETUP")
print("="*80)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=config.LEARNING_RATE,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=config.WEIGHT_DECAY
)

total_steps = config.NUM_EPOCHS * len(train_loader) // config.GRADIENT_ACCUMULATION_STEPS
warmup_steps = int(config.WARMUP_RATIO * total_steps)

def get_lr(step):
    """Cosine learning rate schedule with warmup"""
    if step < warmup_steps:
        return config.LEARNING_RATE * (step + 1) / warmup_steps
    if step >= total_steps:
        return config.LEARNING_RATE * 0.1
    decay_ratio = (step - warmup_steps) / (total_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return config.LEARNING_RATE * 0.1 + coeff * (config.LEARNING_RATE * 0.9)

print(f"✓ Optimizer: AdamW")
print(f"✓ Learning rate: {config.LEARNING_RATE}")
print(f"✓ Weight decay: {config.WEIGHT_DECAY}")
print(f"✓ Total training steps: {total_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Gradient accumulation: {config.GRADIENT_ACCUMULATION_STEPS}")
print(f"✓ Effective batch size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}")

# ============================================================================
# CSV LOGGING SETUP
# ============================================================================

csv_log_path = os.path.join(config.OUTPUT_DIR, "results", "training_log.csv")
with open(csv_log_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['step', 'epoch', 'train_loss', 'eval_loss', 'learning_rate', 'grad_norm'])

print(f"✓ CSV logging initialized: {csv_log_path}")

# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_loss(model, dataloader):
    """Compute average loss on evaluation set"""
    model.eval()
    total_loss = 0.0
    count = 0
    
    with torch.no_grad():
        for batch in dataloader:
            try:
                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].to(device)
                
                loss, _ = model(input_ids, labels)
                if not torch.isnan(loss) and not torch.isinf(loss):
                    total_loss += loss.item()
                    count += 1
            except RuntimeError as e:
                print(f"  ⚠️  Error during evaluation: {e}")
                continue
    
    model.train()
    return total_loss / count if count > 0 else float('nan')

# ============================================================================
# TRAINING LOOP WITH EARLY STOPPING
# ============================================================================

print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)

global_step = 0
best_eval_loss = float('inf')
history = []
consecutive_nans = 0
early_stopping_counter = 0  # NEW: Track epochs without improvement

for epoch in range(config.NUM_EPOCHS):
    print(f"\n{'='*80}")
    print(f"EPOCH {epoch + 1}/{config.NUM_EPOCHS}")
    print(f"{'='*80}")
    
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()
    batches_processed = 0
    
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        loss, _ = model(input_ids, labels)
        
        # Check for NaN or Inf
        if torch.isnan(loss) or torch.isinf(loss):
            consecutive_nans += 1
            print(f"  ⚠️  {'NaN' if torch.isnan(loss) else 'Inf'} loss at step {global_step} (consecutive: {consecutive_nans})")
            
            if consecutive_nans >= config.MAX_CONSECUTIVE_NANS:
                print(f"  ❌ Too many consecutive NaN/Inf losses. Stopping training.")
                break
            
            optimizer.zero_grad()
            continue
        
        # Reset NaN counter on successful batch
        consecutive_nans = 0
        
        # Backward pass with gradient accumulation
        loss = loss / config.GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        
        epoch_loss += loss.item()
        batches_processed += 1
        
        # Optimizer step after accumulation
        if (batch_idx + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
            # Gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config.MAX_GRAD_NORM)
            
            # Update learning rate
            lr = get_lr(global_step)
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr
            
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            
            current_loss = loss.item() * config.GRADIENT_ACCUMULATION_STEPS
            
            # CSV Logging every LOG_STEPS
            if global_step % config.LOG_STEPS == 0:
                # Compute eval loss only at EVAL_STEPS
                eval_loss = evaluate_loss(model, test_loader) if global_step % config.EVAL_STEPS == 0 else None
                
                # Write to CSV
                with open(csv_log_path, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        global_step,
                        epoch + 1,
                        f"{current_loss:.6f}",
                        f"{eval_loss:.6f}" if eval_loss is not None else "",
                        f"{lr:.2e}",
                        f"{grad_norm:.4f}"
                    ])
                
                print(f"  Step {global_step:4d} | Loss: {current_loss:.4f} | LR: {lr:.2e} | Grad: {grad_norm:.2f}")
            
            # Full evaluation at EVAL_STEPS
            if global_step % config.EVAL_STEPS == 0:
                print(f"\n  {'─'*76}")
                print(f"  EVALUATION AT STEP {global_step}")
                print(f"  {'─'*76}")
                
                eval_loss = evaluate_loss(model, test_loader)
                print(f"  → Eval loss: {eval_loss:.4f}")
                
                if eval_loss < best_eval_loss:
                    best_eval_loss = eval_loss
                    print(f"  → ✨ New best eval loss!")
                    
                    # Save best model
                    best_model_path = os.path.join(config.OUTPUT_DIR, "best_model.pt")
                    torch.save({
                        'model': model.state_dict(),
                        'config': model.config,
                        'optimizer': optimizer.state_dict(),
                        'epoch': epoch + 1,
                        'global_step': global_step,
                        'eval_loss': eval_loss,
                    }, best_model_path)
                    print(f"  → Saved best model: {best_model_path}")
                
                print(f"  {'─'*76}\n")
        
        # Clear CUDA cache periodically
        if batch_idx % 100 == 0:
            torch.cuda.empty_cache()
    
    # Check if training was stopped due to NaNs
    if consecutive_nans >= config.MAX_CONSECUTIVE_NANS:
        print(f"\n❌ Training stopped early at epoch {epoch + 1} due to instability")
        break
    
    # End of epoch evaluation
    print(f"\n{'-'*80}")
    print(f"EPOCH {epoch + 1} SUMMARY")
    print(f"{'-'*80}")
    
    avg_train_loss = (epoch_loss * config.GRADIENT_ACCUMULATION_STEPS) / batches_processed if batches_processed > 0 else float('nan')
    eval_loss = evaluate_loss(model, test_loader)
    
    print(f"Average train loss: {avg_train_loss:.4f}")
    print(f"Evaluation loss:    {eval_loss:.4f}")
    
    # Early stopping check
    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        early_stopping_counter = 0
        print(f"✨ Best eval loss improved!")
    else:
        early_stopping_counter += 1
        print(f"⚠️  No improvement for {early_stopping_counter} epoch(s)")
        
        if early_stopping_counter >= config.EARLY_STOPPING_PATIENCE:
            print(f"\n🛑 Early stopping triggered! No improvement for {config.EARLY_STOPPING_PATIENCE} epochs.")
            print(f"   Best eval loss: {best_eval_loss:.4f}")
            break
    
    # Save epoch checkpoint
    checkpoint_path = os.path.join(config.OUTPUT_DIR, f"epoch_{epoch + 1}.pt")
    torch.save({
        'model': model.state_dict(),
        'config': model.config,
        'optimizer': optimizer.state_dict(),
        'epoch': epoch + 1,
        'global_step': global_step,
        'train_loss': avg_train_loss,
        'eval_loss': eval_loss,
    }, checkpoint_path)
    print(f"\n✓ Checkpoint saved: {checkpoint_path}")
    
    # Update history
    history.append({
        'epoch': epoch + 1,
        'global_step': global_step,
        'train_loss': avg_train_loss,
        'eval_loss': eval_loss,
    })
    
    print(f"{'='*80}\n")

# Save final training summary
summary_path = os.path.join(config.OUTPUT_DIR, "results", "training_summary.json")
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump({
        'timestamp': datetime.now().isoformat(),
        'configuration': {
            'freeze_embeddings': config.FREEZE_EMBEDDINGS,
            'freeze_blocks': config.FREEZE_BLOCKS,
            'trainable_params': trainable,
            'total_params': total,
            'trainable_percentage': 100 * trainable / total,
            'num_epochs': config.NUM_EPOCHS,
            'batch_size': config.BATCH_SIZE,
            'gradient_accumulation': config.GRADIENT_ACCUMULATION_STEPS,
            'learning_rate': config.LEARNING_RATE,
            'warmup_ratio': config.WARMUP_RATIO,
            'early_stopping_patience': config.EARLY_STOPPING_PATIENCE,
        },
        'training_history': history,
        'best_eval_loss': best_eval_loss,
        'early_stopped': early_stopping_counter >= config.EARLY_STOPPING_PATIENCE,
    }, f, indent=2, ensure_ascii=False)

print(f"\n✓ Training summary saved: {summary_path}")
print(f"✓ CSV log saved: {csv_log_path}")
print("\n✓ CELL 3 COMPLETE: Training finished")

In [ ]:
# ============================================================================
# CELL 4: EVALUATION & GENERATION
# ============================================================================

import re
from rouge_score import rouge_scorer
from rouge_score.tokenizers import Tokenizer

print("\n" + "="*80)
print("LOADING BEST MODEL FOR EVALUATION")
print("="*80)

# Load best model checkpoint
best_model_path = os.path.join(config.OUTPUT_DIR, "best_model.pt")
best_checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint['model'])
model.eval()

print(f"✓ Best model loaded from step {best_checkpoint['global_step']}")
print(f"✓ Best eval loss: {best_checkpoint['eval_loss']:.4f}")

# ============================================================================
# NEPALI TOKENIZER FOR ROUGE - ENHANCED
# ============================================================================

class NepaliTokenizer(Tokenizer):
    """Enhanced tokenizer for Nepali ROUGE scoring - strict Devanagari only"""
    def tokenize(self, text):
        # Remove Devanagari danda
        text = text.replace("।", "")
        # Normalize Devanagari numbers to generic NUM token
        text = re.sub(r"[०-९]+", "NUM", text)
        # Keep ONLY Devanagari characters and whitespace (removes English, punctuation, etc.)
        text = re.sub(r"[^\u0900-\u097F\s]", "", text)
        return text.split()

nepali_tokenizer = NepaliTokenizer()
rouge_scorer_nepali = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    tokenizer=nepali_tokenizer,
    use_stemmer=False
)

# ============================================================================
# GENERATION FUNCTION WITH TOP-K AND TOP-P SAMPLING
# ============================================================================

def generate_summary(model, tokenizer, text, max_new_tokens=None, top_k=None, top_p=None, temperature=None):
    """
    Generate summary using top-k and top-p (nucleus) sampling
    
    Args:
        model: GPT model
        tokenizer: SentencePiece tokenizer
        text: Input article text
        max_new_tokens: Maximum tokens to generate (default: config.MAX_NEW_TOKENS)
        top_k: Top-k sampling parameter (default: config.GENERATION_TOP_K)
        top_p: Top-p (nucleus) sampling parameter (default: config.GENERATION_TOP_P)
        temperature: Sampling temperature (default: config.GENERATION_TEMPERATURE)
    
    Returns:
        Generated summary string
    """
    if max_new_tokens is None:
        max_new_tokens = config.MAX_NEW_TOKENS
    if top_k is None:
        top_k = config.GENERATION_TOP_K
    if top_p is None:
        top_p = config.GENERATION_TOP_P
    if temperature is None:
        temperature = config.GENERATION_TEMPERATURE
    
    model.eval()
    prompt = PROMPT_TEMPLATE.format(text=text)
    tokens = tokenizer.encode(prompt)
    input_ids = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    
    generated = input_ids.clone()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            if generated.size(1) >= model.config.block_size:
                break
            
            _, logits = model(generated)
            logits = logits[:, -1, :] / temperature
            
            # Top-k filtering
            top_k_logits, top_k_indices = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1)
            
            # Top-p (nucleus) filtering
            sorted_logits, sorted_indices = torch.sort(top_k_logits, descending=True, dim=-1)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            
            # Remove tokens with cumulative probability above the threshold
            sorted_indices_to_remove = cumulative_probs > top_p
            # Keep at least one token
            sorted_indices_to_remove[..., 0] = False
            
            # Create filtered logits
            filtered_logits = sorted_logits.clone()
            filtered_logits[sorted_indices_to_remove] = float('-inf')
            
            # Sample from filtered distribution
            probs = F.softmax(filtered_logits, dim=-1)
            next_token_idx = torch.multinomial(probs, 1)
            next_token = torch.gather(sorted_indices, -1, next_token_idx)
            next_token = torch.gather(top_k_indices, -1, next_token)
            
            generated = torch.cat([generated, next_token], dim=1)
            
            # Stop at EOS or padding token
            if next_token.item() == 0:
                break
    
    summary = tokenizer.decode(generated[0, len(tokens):].tolist())
    return summary

# ============================================================================
# COMPUTE ROUGE ON SAMPLE TEST SET
# ============================================================================

print("\n" + "="*80)
print(f"COMPUTING ROUGE SCORES ON {config.ROUGE_SAMPLE_SIZE} TEST SAMPLES")
print("="*80)

rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}

print(f"Generating summaries for {config.ROUGE_SAMPLE_SIZE} test samples...")
print(f"Generation params: max_tokens={config.MAX_NEW_TOKENS}, top_k={config.GENERATION_TOP_K}, top_p={config.GENERATION_TOP_P}, temp={config.GENERATION_TEMPERATURE}")

sample_size = min(config.ROUGE_SAMPLE_SIZE, len(test_dataset))

for i in tqdm(range(sample_size), desc="ROUGE Evaluation"):
    item = test_dataset.data[i]
    
    # Generate summary
    pred_summary = generate_summary(
        model, 
        sp, 
        item['text'],
        max_new_tokens=config.MAX_NEW_TOKENS,
        top_k=config.GENERATION_TOP_K,
        top_p=config.GENERATION_TOP_P,
        temperature=config.GENERATION_TEMPERATURE
    )
    
    ref_summary = item['summary']
    
    # Compute ROUGE
    result = rouge_scorer_nepali.score(ref_summary, pred_summary)
    rouge_scores['rouge1'].append(result['rouge1'].fmeasure)
    rouge_scores['rouge2'].append(result['rouge2'].fmeasure)
    rouge_scores['rougeL'].append(result['rougeL'].fmeasure)

# Calculate averages
avg_rouge_sample = {k: sum(v) / len(v) * 100 for k, v in rouge_scores.items()}

print(f"\n{'='*80}")
print(f"ROUGE SCORES (on {sample_size} samples)")
print(f"{'='*80}")
print(f"ROUGE-1: {avg_rouge_sample['rouge1']:.2f}")
print(f"ROUGE-2: {avg_rouge_sample['rouge2']:.2f}")
print(f"ROUGE-L: {avg_rouge_sample['rougeL']:.2f}")

# ============================================================================
# COMPUTE ROUGE ON FULL TEST SET
# ============================================================================

print("\n" + "="*80)
print("COMPUTING ROUGE SCORES ON FULL TEST SET")
print("="*80)

rouge_scores_full = {'rouge1': [], 'rouge2': [], 'rougeL': []}

print(f"Generating summaries for {len(test_dataset)} test samples...")

for i in tqdm(range(len(test_dataset)), desc="Full ROUGE Evaluation"):
    item = test_dataset.data[i]
    
    # Generate summary
    pred_summary = generate_summary(
        model, 
        sp, 
        item['text'],
        max_new_tokens=config.MAX_NEW_TOKENS,
        top_k=config.GENERATION_TOP_K,
        top_p=config.GENERATION_TOP_P,
        temperature=config.GENERATION_TEMPERATURE
    )
    
    ref_summary = item['summary']
    
    # Compute ROUGE
    result = rouge_scorer_nepali.score(ref_summary, pred_summary)
    rouge_scores_full['rouge1'].append(result['rouge1'].fmeasure)
    rouge_scores_full['rouge2'].append(result['rouge2'].fmeasure)
    rouge_scores_full['rougeL'].append(result['rougeL'].fmeasure)

# Calculate averages
avg_rouge_full = {k: sum(v) / len(v) * 100 for k, v in rouge_scores_full.items()}

print(f"\n{'='*80}")
print("FINAL ROUGE SCORES (FULL TEST SET)")
print(f"{'='*80}")
print(f"ROUGE-1: {avg_rouge_full['rouge1']:.2f}")
print(f"ROUGE-2: {avg_rouge_full['rouge2']:.2f}")
print(f"ROUGE-L: {avg_rouge_full['rougeL']:.2f}")

# Save ROUGE results
rouge_results_path = os.path.join(config.OUTPUT_DIR, "results", "rouge_scores.json")
with open(rouge_results_path, 'w', encoding='utf-8') as f:
    json.dump({
        'timestamp': datetime.now().isoformat(),
        'test_samples': len(test_dataset),
        'generation_params': {
            'max_new_tokens': config.MAX_NEW_TOKENS,
            'top_k': config.GENERATION_TOP_K,
            'top_p': config.GENERATION_TOP_P,
            'temperature': config.GENERATION_TEMPERATURE,
        },
        'rouge_scores_sample': {
            'sample_size': sample_size,
            'scores': avg_rouge_sample,
        },
        'rouge_scores_full': avg_rouge_full,
        'best_model_step': best_checkpoint['global_step'],
        'best_eval_loss': best_checkpoint['eval_loss'],
    }, f, indent=2, ensure_ascii=False)

print(f"✓ ROUGE scores saved: {rouge_results_path}")

# ============================================================================
# SIDE-BY-SIDE COMPARISON (10 SAMPLES)
# ============================================================================

print("\n" + "="*80)
print("SIDE-BY-SIDE COMPARISON (10 SAMPLES)")
print("="*80)

num_samples = min(10, len(test_dataset))
comparison_results = []

for i in range(num_samples):
    item = test_dataset.data[i]
    
    # Generate summary
    pred_summary = generate_summary(
        model, 
        sp, 
        item['text'],
        max_new_tokens=config.MAX_NEW_TOKENS,
        top_k=config.GENERATION_TOP_K,
        top_p=config.GENERATION_TOP_P,
        temperature=config.GENERATION_TEMPERATURE
    )
    
    ref_summary = item['summary']
    
    # Compute ROUGE for this example
    result = rouge_scorer_nepali.score(ref_summary, pred_summary)
    
    print(f"\n{'-'*80}")
    print(f"EXAMPLE {i+1}")
    print(f"{'-'*80}")
    print(f"\nARTICLE (first 200 chars):")
    print(item['text'][:200] + "...")
    print(f"\nREFERENCE SUMMARY:")
    print(ref_summary)
    print(f"\nGENERATED SUMMARY:")
    print(pred_summary)
    print(f"\nROUGE SCORES:")
    print(f"  R-1: {result['rouge1'].fmeasure*100:.2f}")
    print(f"  R-2: {result['rouge2'].fmeasure*100:.2f}")
    print(f"  R-L: {result['rougeL'].fmeasure*100:.2f}")
    
    comparison_results.append({
        'article_preview': item['text'][:200],
        'reference_summary': ref_summary,
        'generated_summary': pred_summary,
        'rouge1': result['rouge1'].fmeasure * 100,
        'rouge2': result['rouge2'].fmeasure * 100,
        'rougeL': result['rougeL'].fmeasure * 100,
    })

# Save comparison examples
comparison_path = os.path.join(config.OUTPUT_DIR, "results", "comparison_examples.json")
with open(comparison_path, 'w', encoding='utf-8') as f:
    json.dump(comparison_results, f, indent=2, ensure_ascii=False)

print(f"\n✓ Comparison examples saved: {comparison_path}")

# ============================================================================
# CUSTOM GENERATION FUNCTION
# ============================================================================

def generate_custom_summary(text, max_tokens=128, top_k=50, top_p=0.9, temperature=0.5):
    """
    Convenience function for generating summaries with custom parameters
    
    Usage:
        summary = generate_custom_summary(
            "Your Nepali article text here...",
            max_tokens=150,
            top_k=60,
            top_p=0.95,
            temperature=0.6
        )
    
    Args:
        text: Input Nepali article
        max_tokens: Maximum tokens to generate (default: 128)
        top_k: Top-k sampling (default: 50)
        top_p: Nucleus sampling threshold (default: 0.9)
        temperature: Sampling temperature (default: 0.5)
    
    Returns:
        Generated summary string
    """
    return generate_summary(model, sp, text, max_tokens, top_k, top_p, temperature)

print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)
print("\nYou can now generate custom summaries using:")
print("  generate_custom_summary(text, max_tokens=128, top_k=50, top_p=0.9, temperature=0.5)")
print("\nExample:")
print('  summary = generate_custom_summary("आफ्नो लेख यहाँ...", max_tokens=100, temperature=0.3)')
print("\n✓ CELL 4 COMPLETE: Evaluation and generation ready")